# 🚨 SLA Breach Predictor — Priority-Aware XGBoost + AWS S3 & Lambda

> **Predict IT incident SLA breaches before they happen — and deploy the model to AWS for real-time scoring.**

---

## 📐 End-to-End Architecture

```
Incident Event Log (119k rows)
         │
         ▼
┌──────────────────────┐
│  ETL & Feature Eng.  │  → resolution_hours, sla_target, sla_breach label
└──────────┬───────────┘
           │
           ▼
┌──────────────────────┐
│  XGBoost Classifier  │  → AUC: 0.8218  |  F1: 0.8351  |  MLflow tracked
└──────────┬───────────┘
           │
           ▼
┌──────────────────────┐
│  AWS S3 Bucket       │  → model.pkl + encoders.pkl + model_evaluation.png
└──────────┬───────────┘
           │
           ▼
┌──────────────────────┐
│  AWS Lambda Function │  → Real-time SLA breach scoring via HTTP event
└──────────────────────┘
```

## 🎯 SLA Targets by Priority

| Priority | SLA Window | Breach Rate |
|---|---|---|
| 1 - Critical | 4 hours | 88.3% |
| 2 - High | 8 hours | 79.5% |
| 3 - Moderate | 24 hours | 65.3% |
| 4 - Low | 72 hours | 49.7% |

## 📦 Dataset
[Incident Response Log — Vipul Shinde](https://www.kaggle.com/datasets/vipulshinde/incident-response-log)

---
## ⚙️ Section 1 — Install & Import Dependencies

In [ ]:
# Kaggle has most packages pre-installed; install any extras
!pip install -q mlflow boto3

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import mlflow
import pickle
import json
import os
import boto3
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, precision_recall_curve
)
from sklearn.preprocessing import LabelEncoder
from IPython.display import FileLink

print(f'✅ XGBoost  : {xgb.__version__}')
print(f'✅ MLflow   : {mlflow.__version__}')
print(f'✅ boto3    : {boto3.__version__}')

---
## 📂 Section 2 — Load & Inspect Data

In [ ]:
df = pd.read_csv(
    '/kaggle/input/incident-response-log/incident_event_log.csv',
    encoding='latin1'
)
print(f'Loaded {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head(3)

In [ ]:
# Quick data quality check
key_cols = ['opened_at', 'closed_at', 'resolved_at',
            'priority', 'category', 'impact', 'urgency']
print('Null counts in key columns:')
print(df[key_cols].isnull().sum())
print('\nPriority distribution:')
print(df['priority'].value_counts())

---
## 🔧 Section 3 — Feature Engineering & SLA Label Creation

In [ ]:
# ── Parse timestamps ───────────────────────────────────────────────────────────
df['opened_at']   = pd.to_datetime(df['opened_at'],   errors='coerce', dayfirst=True)
df['closed_at']   = pd.to_datetime(df['closed_at'],   errors='coerce', dayfirst=True)
df['resolved_at'] = pd.to_datetime(df['resolved_at'], errors='coerce', dayfirst=True)

df['resolution_hours'] = (
    df['resolved_at'] - df['opened_at']
).dt.total_seconds() / 3600

print('Resolution time statistics (hours):')
print(df['resolution_hours'].describe().round(2))

In [ ]:
# ── Priority-aware SLA targets ─────────────────────────────────────────────────
SLA_TARGETS = {
    '1 - Critical': 4,    # 4 hours
    '2 - High':     8,    # 8 hours
    '3 - Moderate': 24,   # 24 hours
    '4 - Low':      72,   # 72 hours
}

df['sla_target_hours'] = df['priority'].astype(str).str.strip().map(SLA_TARGETS)
df['sla_breach']       = (df['resolution_hours'] > df['sla_target_hours']).astype(int)

print(f'Overall breach rate : {df["sla_breach"].mean():.1%}')
print('\nClass distribution:')
print(df['sla_breach'].value_counts().rename({0: 'No Breach', 1: 'Breach'}))
print('\nBreach rate by priority:')
print(df.groupby('priority')['sla_breach'].mean().round(3))

In [ ]:
# ── Encode categorical features ────────────────────────────────────────────────
FEATURES = ['priority', 'category', 'impact', 'urgency',
            'assignment_group', 'opened_by', 'contact_type']

df_model = df[FEATURES + ['sla_breach']].dropna().copy()

# Save encoders so Lambda can apply the same encoding at inference time
encoders = {}
for col in FEATURES:
    if df_model[col].dtype == 'object':
        le = LabelEncoder()
        df_model[col] = le.fit_transform(df_model[col].astype(str))
        encoders[col] = le

X = df_model[FEATURES]
y = df_model['sla_breach']

print(f'Feature matrix: {X.shape[0]:,} rows × {X.shape[1]} features')
print(f'Features: {FEATURES}')

---
## ✂️ Section 4 — Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train : {X_train.shape[0]:,} samples  |  Breach rate: {y_train.mean():.1%}')
print(f'Test  : {X_test.shape[0]:,} samples  |  Breach rate: {y_test.mean():.1%}')

---
## 🤖 Section 5 — Train XGBoost Model (MLflow Tracked)

In [ ]:
mlflow.start_run(run_name='xgboost-sla-priority-aware')

PARAMS = {
    'n_estimators':     200,
    'max_depth':        6,
    'learning_rate':    0.1,
    'subsample':        0.8,
    'colsample_bytree': 0.8,
    'random_state':     42,
    'eval_metric':      'auc'
}

model = xgb.XGBClassifier(**PARAMS)
model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=50
)

train_auc = model.evals_result()['validation_0']['auc']
test_auc  = model.evals_result()['validation_1']['auc']

print(f'\n✅ Training complete')
print(f'   Final Train AUC : {train_auc[-1]:.4f}')
print(f'   Final Test  AUC : {test_auc[-1]:.4f}')

---
## 📊 Section 6 — Evaluate & Find Optimal Threshold

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]
auc    = roc_auc_score(y_test, y_prob)

print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['No Breach', 'Breach']))
print(f'AUC Score: {auc:.4f}')

mlflow.log_params(PARAMS)
mlflow.log_metric('auc', auc)
mlflow.end_run()

In [ ]:
# ── Find optimal decision threshold (maximise F1) ──────────────────────────────
precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob)
f1_scores   = 2 * precisions * recalls / (precisions + recalls + 1e-9)
best_idx    = np.argmax(f1_scores)
best_thresh = thresholds[best_idx]

print(f'Default threshold : 0.500')
print(f'Optimal threshold : {best_thresh:.3f}')
print(f'Best F1 Score     : {f1_scores[best_idx]:.4f}')
print('\n💡 Using the optimal threshold improves Breach class F1 vs default 0.5')

---
## 📈 Section 7 — Evaluation Plots

In [ ]:
BLUE   = '#2E75B6'
ORANGE = '#E06C2A'
GREEN  = '#2E8B57'
RED    = '#CC2936'

fig = plt.figure(figsize=(24, 16))
fig.suptitle('SLA Breach Predictor — Priority-Aware XGBoost Evaluation',
             fontsize=16, fontweight='bold', y=0.98)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# Plot 1: Breach rate by priority
ax1 = fig.add_subplot(gs[0, 0])
priority_breach = df.groupby('priority')['sla_breach'].mean() * 100
priority_labels = [f"{p}\n(target={SLA_TARGETS[p]}h)" for p in priority_breach.index]
bars = ax1.bar(priority_labels, priority_breach.values,
               color=[RED if v > 50 else ORANGE if v > 20 else GREEN
                      for v in priority_breach.values],
               edgecolor='white', width=0.5)
ax1.set_title('Breach Rate by Priority', fontweight='bold')
ax1.set_ylabel('Breach Rate (%)')
ax1.set_ylim(0, 110)
for bar, val in zip(bars, priority_breach.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
             f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax1.spines[['top', 'right']].set_visible(False)

# Plot 2: Learning curve
ax2 = fig.add_subplot(gs[0, 1])
rounds = range(len(train_auc))
ax2.plot(rounds, train_auc, color=BLUE,   label='Train AUC',      linewidth=2)
ax2.plot(rounds, test_auc,  color=ORANGE, label='Validation AUC', linewidth=2)
ax2.set_title('Learning Curve (AUC per Round)', fontweight='bold')
ax2.set_xlabel('Boosting Round')
ax2.set_ylabel('AUC')
ax2.legend()
ax2.annotate(f'Final: {test_auc[-1]:.4f}',
             xy=(len(test_auc)-1, test_auc[-1]),
             xytext=(-60, -20), textcoords='offset points',
             arrowprops=dict(arrowstyle='->', color='black'), fontsize=9)
ax2.spines[['top', 'right']].set_visible(False)

# Plot 3: Confusion matrix
ax3 = fig.add_subplot(gs[0, 2])
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['No Breach', 'Breach']).plot(
    ax=ax3, colorbar=False, cmap='Blues')
ax3.set_title('Confusion Matrix', fontweight='bold')

# Plot 4: ROC curve
ax4 = fig.add_subplot(gs[1, 0])
RocCurveDisplay.from_predictions(y_test, y_prob, ax=ax4,
                                  name=f'XGBoost (AUC = {auc:.3f})',
                                  color=BLUE, linewidth=2)
ax4.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random classifier')
ax4.set_title('ROC Curve', fontweight='bold')
ax4.legend(loc='lower right')
ax4.spines[['top', 'right']].set_visible(False)

# Plot 5: Precision / Recall / F1 vs Threshold
ax5 = fig.add_subplot(gs[1, 1])
ax5.plot(thresholds, precisions[:-1], color=BLUE,   label='Precision', linewidth=2)
ax5.plot(thresholds, recalls[:-1],    color=ORANGE,  label='Recall',    linewidth=2)
ax5.plot(thresholds, f1_scores[:-1],  color=GREEN,   label='F1 Score',  linewidth=2)
ax5.axvline(best_thresh, color=RED, linestyle='--', linewidth=1.5,
            label=f'Optimal = {best_thresh:.2f}')
ax5.axvline(0.5, color='grey', linestyle=':', linewidth=1.5, label='Default = 0.5')
ax5.set_title('Precision / Recall / F1 vs Threshold', fontweight='bold')
ax5.set_xlabel('Decision Threshold')
ax5.set_ylabel('Score')
ax5.legend(fontsize=8)
ax5.spines[['top', 'right']].set_visible(False)

# Plot 6: Feature importance
ax6 = fig.add_subplot(gs[1, 2])
importances = model.feature_importances_
sorted_idx  = np.argsort(importances)
colors = [ORANGE if i == sorted_idx[-1] else BLUE for i in sorted_idx]
ax6.barh([FEATURES[i] for i in sorted_idx], importances[sorted_idx],
          color=colors, edgecolor='white')
ax6.set_title('Feature Importance\n(top feature in orange)', fontweight='bold')
ax6.set_xlabel('Importance Score')
for i, idx in enumerate(sorted_idx):
    ax6.text(importances[idx] + 0.002, i,
             f'{importances[idx]:.3f}', va='center', fontsize=9)
ax6.spines[['top', 'right']].set_visible(False)

plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('✅ Saved model_evaluation.png')

---
## 💾 Section 8 — Save Model Artefacts

In [ ]:
# Save trained model
with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)

# Save label encoders (needed by Lambda for inference)
with open('encoders.pkl', 'wb') as f:
    pickle.dump(encoders, f)

# Save metadata
metadata = {
    'features':        FEATURES,
    'sla_targets':     SLA_TARGETS,
    'optimal_thresh':  round(float(best_thresh), 4),
    'auc':             round(float(auc), 4),
    'best_f1':         round(float(f1_scores[best_idx]), 4),
    'train_samples':   int(X_train.shape[0]),
    'test_samples':    int(X_test.shape[0]),
}
with open('metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('✅ Artefacts saved:')
print('   model.pkl        — trained XGBoost model')
print('   encoders.pkl     — label encoders for all categorical features')
print('   metadata.json    — model config, threshold, and metrics')
print('   model_evaluation.png — evaluation dashboard')
print(f'\n   AUC             : {auc:.4f}')
print(f'   Optimal thresh  : {best_thresh:.3f}')
print(f'   SLA targets     : {SLA_TARGETS}')

---
## ☁️ Section 9 — Upload Artefacts to AWS S3

> **Setup:** Add your AWS credentials as Kaggle Secrets (`AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`) and set your bucket name below.
>
> In Kaggle: *Add-ons → Secrets → Add Secret*

In [ ]:
from kaggle_secrets import UserSecretsClient

# ── Configuration — update these ──────────────────────────────────────────────
S3_BUCKET = 'your-sla-predictor-bucket'   # ← replace with your bucket name
S3_PREFIX = 'sla-breach-predictor/v1'     # folder path inside the bucket
AWS_REGION = 'eu-central-1'               # ← replace with your region

# ── Load credentials from Kaggle Secrets ──────────────────────────────────────
secrets = UserSecretsClient()
aws_access_key = secrets.get_secret('AWS_ACCESS_KEY_ID')
aws_secret_key = secrets.get_secret('AWS_SECRET_ACCESS_KEY')

s3 = boto3.client(
    's3',
    region_name=AWS_REGION,
    aws_access_key_id=aws_access_key,
    aws_secret_access_key=aws_secret_key
)

print(f'✅ Connected to AWS S3 | Region: {AWS_REGION} | Bucket: {S3_BUCKET}')

In [ ]:
# ── Upload all artefacts to S3 ─────────────────────────────────────────────────
artefacts = [
    ('model.pkl',            'application/octet-stream'),
    ('encoders.pkl',         'application/octet-stream'),
    ('metadata.json',        'application/json'),
    ('model_evaluation.png', 'image/png'),
]

for filename, content_type in artefacts:
    s3_key = f'{S3_PREFIX}/{filename}'
    s3.upload_file(
        filename,
        S3_BUCKET,
        s3_key,
        ExtraArgs={'ContentType': content_type}
    )
    print(f'✅ Uploaded  s3://{S3_BUCKET}/{s3_key}')

print('\n🎉 All artefacts uploaded to S3 successfully!')

In [ ]:
# ── Verify uploads by listing S3 prefix ───────────────────────────────────────
response = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=S3_PREFIX)
print(f'Files in s3://{S3_BUCKET}/{S3_PREFIX}/')
print('-' * 60)
for obj in response.get('Contents', []):
    size_kb = obj['Size'] / 1024
    print(f"  {obj['Key'].split('/')[-1]:<30} {size_kb:>8.1f} KB")

---
## ⚡ Section 10 — AWS Lambda Function for Real-Time Inference

The cell below contains the **complete Lambda handler** to deploy.  
It loads the model and encoders from S3 on cold start, then scores new incidents in real time.

### Deploy steps
1. Create a Lambda function in the AWS Console (Python 3.11 runtime)
2. Add an IAM role with `s3:GetObject` permission on your bucket
3. Set environment variables: `S3_BUCKET`, `S3_PREFIX`, `OPTIMAL_THRESHOLD`
4. Paste the code below into the Lambda editor (or zip + upload)
5. Set memory to **512 MB** and timeout to **30 s**
6. (Optional) Add a **Function URL** or **API Gateway** trigger for HTTP access

In [ ]:
lambda_handler_code = '''
# ─────────────────────────────────────────────────────────────────────────────
# lambda_function.py  —  SLA Breach Predictor
# Deploy this file to AWS Lambda (Python 3.11 runtime, 512 MB, 30s timeout)
#
# Required Lambda environment variables:
#   S3_BUCKET          e.g. "your-sla-predictor-bucket"
#   S3_PREFIX          e.g. "sla-breach-predictor/v1"
#   OPTIMAL_THRESHOLD  e.g. "0.433"
#
# Required Lambda layers / packages:
#   xgboost, scikit-learn (use an AWS-provided layer or build a zip)
# ─────────────────────────────────────────────────────────────────────────────
import json
import os
import pickle
import boto3
import numpy as np

# ── Module-level cache (persists across warm invocations) ─────────────────────
_model    = None
_encoders = None
_metadata = None

S3_BUCKET  = os.environ['S3_BUCKET']
S3_PREFIX  = os.environ['S3_PREFIX']
THRESHOLD  = float(os.environ.get('OPTIMAL_THRESHOLD', '0.433'))
FEATURES   = ['priority', 'category', 'impact', 'urgency',
               'assignment_group', 'opened_by', 'contact_type']

s3 = boto3.client('s3')


def _load_artefacts():
    """Download model + encoders from S3 on cold start."""
    global _model, _encoders, _metadata
    if _model is not None:
        return  # already loaded (warm start)

    def _get(key):
        return s3.get_object(Bucket=S3_BUCKET,
                             Key=f'{S3_PREFIX}/{key}')["Body"].read()

    _model    = pickle.loads(_get('model.pkl'))
    _encoders = pickle.loads(_get('encoders.pkl'))
    _metadata = json.loads(_get('metadata.json'))
    print(f'[COLD START] Model loaded | AUC={_metadata["auc"]} | '
          f'threshold={_metadata["optimal_thresh"]}')


def _encode_input(payload: dict) -> np.ndarray:
    """Apply label encoding to a single incident dict."""
    row = []
    for col in FEATURES:
        val = str(payload.get(col, 'unknown'))
        if col in _encoders:
            le = _encoders[col]
            # Handle unseen categories gracefully
            val = val if val in le.classes_ else le.classes_[0]
            row.append(le.transform([val])[0])
        else:
            row.append(float(val))
    return np.array(row).reshape(1, -1)


def lambda_handler(event, context):
    """
    Expected event payload (single incident):
    {
        "priority":         "2 - High",
        "category":         "Network",
        "impact":           "2 - Medium",
        "urgency":          "2 - Medium",
        "assignment_group": "Network Operations",
        "opened_by":        "agent_42",
        "contact_type":     "Email"
    }

    Returns:
    {
        "sla_breach_predicted": true,
        "breach_probability":   0.81,
        "threshold_used":       0.433,
        "risk_level":           "HIGH",
        "sla_target_hours":     8,
        "recommendation":       "Escalate immediately — high breach probability"
    }
    """
    try:
        _load_artefacts()

        # Support both direct invocation and API Gateway proxy events
        body = event
        if 'body' in event:
            body = json.loads(event['body']) if isinstance(event['body'], str) else event['body']

        X = _encode_input(body)
        prob = float(_model.predict_proba(X)[0, 1])
        breach = prob >= THRESHOLD

        sla_targets = _metadata['sla_targets']
        priority    = str(body.get('priority', ''))
        sla_hours   = sla_targets.get(priority)

        # Risk banding
        if prob >= 0.80:
            risk = 'CRITICAL'
            rec  = 'Escalate immediately — very high breach probability'
        elif prob >= 0.60:
            risk = 'HIGH'
            rec  = 'Prioritise resolution — breach likely without action'
        elif prob >= 0.40:
            risk = 'MEDIUM'
            rec  = 'Monitor closely — breach possible'
        else:
            risk = 'LOW'
            rec  = 'On track — continue normal resolution workflow'

        result = {
            'sla_breach_predicted': bool(breach),
            'breach_probability':   round(prob, 4),
            'threshold_used':       THRESHOLD,
            'risk_level':           risk,
            'sla_target_hours':     sla_hours,
            'recommendation':       rec,
        }

        return {
            'statusCode': 200,
            'headers':    {'Content-Type': 'application/json'},
            'body':       json.dumps(result)
        }

    except Exception as e:
        return {
            'statusCode': 500,
            'body':       json.dumps({'error': str(e)})
        }
'''

# Save Lambda handler locally so it can be zipped and deployed
with open('lambda_function.py', 'w') as f:
    f.write(lambda_handler_code)

print('✅ lambda_function.py written')
print('   → Zip this file (+ xgboost/sklearn layers) and upload to AWS Lambda')

---
## 🧪 Section 11 — Test the Lambda Handler Locally

In [ ]:
# ── Simulate a Lambda invocation locally ──────────────────────────────────────
# (No AWS call needed — uses the in-memory model and encoders)

SLA_TARGETS_MAP = {
    '1 - Critical': 4,
    '2 - High':     8,
    '3 - Moderate': 24,
    '4 - Low':      72,
}

def local_predict(incident: dict, threshold: float = best_thresh) -> dict:
    row = []
    for col in FEATURES:
        val = str(incident.get(col, 'unknown'))
        if col in encoders:
            le = encoders[col]
            val = val if val in le.classes_ else le.classes_[0]
            row.append(le.transform([val])[0])
        else:
            row.append(float(val))
    X_in = np.array(row).reshape(1, -1)
    prob   = float(model.predict_proba(X_in)[0, 1])
    breach = prob >= threshold
    risk   = ('CRITICAL' if prob >= 0.80 else
              'HIGH'     if prob >= 0.60 else
              'MEDIUM'   if prob >= 0.40 else 'LOW')
    return {
        'sla_breach_predicted': breach,
        'breach_probability':   round(prob, 4),
        'threshold_used':       round(threshold, 3),
        'risk_level':           risk,
        'sla_target_hours':     SLA_TARGETS_MAP.get(incident.get('priority'))
    }


test_cases = [
    {
        'label':          '🔴 Critical incident — expect BREACH',
        'priority':       '1 - Critical',
        'category':       'Network',
        'impact':         '1 - High',
        'urgency':        '1 - High',
        'assignment_group': 'Network Operations',
        'opened_by':      'agent_01',
        'contact_type':   'Phone'
    },
    {
        'label':          '🟡 Low-priority ticket — expect NO BREACH',
        'priority':       '4 - Low',
        'category':       'Hardware',
        'impact':         '3 - Low',
        'urgency':        '3 - Low',
        'assignment_group': 'Desktop Support',
        'opened_by':      'agent_99',
        'contact_type':   'Email'
    },
]

for tc in test_cases:
    label = tc.pop('label')
    result = local_predict(tc)
    print(f'\n{label}')
    print(f'  Breach predicted : {result["sla_breach_predicted"]}')
    print(f'  Probability      : {result["breach_probability"]:.1%}')
    print(f'  Risk level       : {result["risk_level"]}')
    print(f'  SLA target       : {result["sla_target_hours"]} hours')

---
## 📋 Section 12 — Results Summary

In [ ]:
print('=' * 55)
print('  SLA BREACH PREDICTOR — FINAL RESULTS')
print('=' * 55)
print(f'  Dataset          : 119,998 incidents')
print(f'  Overall breach rate : {df["sla_breach"].mean():.1%}')
print()
print(f'  Model            : XGBoost (200 estimators, depth 6)')
print(f'  AUC Score        : {auc:.4f}')
print(f'  Best F1 (Breach) : {f1_scores[best_idx]:.4f}')
print(f'  Optimal threshold: {best_thresh:.3f}')
print()
print('  Artefacts in S3:')
print(f'  └── s3://{S3_BUCKET}/{S3_PREFIX}/')
print(f'      ├── model.pkl')
print(f'      ├── encoders.pkl')
print(f'      ├── metadata.json')
print(f'      └── model_evaluation.png')
print()
print('  Lambda function  : lambda_function.py (ready to deploy)')
print('=' * 55)

FileLink('model.pkl')